# 02 - Photo column reference cleaning & deduplication

Demonstrates `utils.resolve_photo_references`, the integrated cleaning and dedup step used to clean the photo columns.

Three methods for cleaning were implemented:
1. **Missing files** -- `photo` references where the file is not available on disk.
2. **Filename duplicates** -- multiple rows reference the same filename.
3. **Content duplicates** -- different filenames, identical bytes (detected by file size + MD5 hashing).

The most likely origin of errors are cell-service/connectivity interruptions during the campaigns.

In case of duplicates the function keeps the row whose `timestamp` is closest to the datetime embedded in the filename. Ties fall back to the row with the lowest `field_id`. Non-winners have their `photo` reference cleared, and redundant on-disk files are deleted for content-duplicate groups. References with missing files are removed.

In [ ]:
from pathlib import Path

import geopandas as gpd

from utils import resolve_photo_references

## Inputs

We start from the `data/` directory containing the published files. Switch `CAMPAIGN` / `CODE` to inspect a different campaign.

In [ ]:
DATA_DIR = Path("../data")
CAMPAIGN = "01_LRS23"
CODE     = "LRS23"

gpkg_path  = DATA_DIR / CAMPAIGN / f"CropHype-Fields-Kenya_{CODE}.gpkg"
image_dir  = DATA_DIR / CAMPAIGN / "DCIM"

gdf = gpd.read_file(gpkg_path)
print(f"{CODE}: {len(gdf)} fields loaded")

## Run cleaning function

In [ ]:
resolve_photo_references(gdf, image_dir, name=CODE)